In [1]:
class QCQPInstance:
    pass
import pickle

with open('instances_5_1_50.pkl', 'rb') as f:
    instances = pickle.load(f)

# Convert to dicts
instances = [inst.__dict__ for inst in instances]




In [2]:
"""
============================================================
FEATURE EXTRACTION - ALL SETUPS (fDD, sDD, DI) - FINAL
============================================================
✅ mu3 is now RAW 3rd central moment (matching MINLPLIB version)
✅ Pruning policy via global switches (matching MINLPLIB version):
      INCLUDE_COND_A0        = False  -> drops log10_cond_A0 by default
      INCLUDE_DIAG_OFF_RATIO = False  -> drops diag_off_ratio by default
      INCLUDE_SCALE_FEATURES = False  -> drops scale by default
      KEEP_LOGCOND_STATS     = ("avg", "IQR", "mu3", "CV")  -> only these stats kept for LogCond
      KEEP_LOGDIAGOFF_STATS  = ("avg", "IQR", "mu3", "CV")  -> only these stats kept for LogDiagOff
      comm_min DROPPED from commutator output (only comm_mean / comm_max kept)
✅ Works with QCQPInstance objects OR dicts
✅ Aligned column ordering with MINLPLIB version
============================================================
"""

import numpy as np
import pandas as pd
import networkx as nx
from networkx.algorithms import bipartite
import pickle
import random
from typing import Any, Dict, Set, List


# ============================================================
# PRUNING POLICY GLOBALS  (edit here to change defaults)
# ============================================================

INCLUDE_COND_A0        = False          # drop log10_cond_A0 per-matrix in A0 slot
INCLUDE_DIAG_OFF_RATIO = False          # drop diag_off_ratio extras by default
INCLUDE_SCALE_FEATURES = False          # drop scale features (heavy / noisy)

KEEP_LOGCOND_STATS    = ("avg", "IQR", "mu3", "CV")   # stats to keep for LogCond
KEEP_LOGDIAGOFF_STATS = ("avg", "IQR", "mu3", "CV")   # stats to keep for LogDiagOff

# Epsilon constants (centralised)
EPS_NUM   = 1e-6
EPS_CV    = 1e-6
EPS_EIG   = 1e-12


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def _get(inst: Any, key: str, default=None):
    """Get attribute from instance or dict"""
    if isinstance(inst, dict):
        return inst.get(key, default)
    return getattr(inst, key, default)


def _to_dict(inst: Any) -> Dict[str, Any]:
    """Convert instance to dict"""
    if isinstance(inst, dict):
        return inst
    if hasattr(inst, "__dict__"):
        return dict(inst.__dict__)
    raise TypeError(f"Unsupported instance type: {type(inst)}")


def _sym(A):
    """Symmetrize matrix"""
    A = np.asarray(A, dtype=float)
    return 0.5 * (A + A.T)


def _safe_signed_div(num: float, den: float, eps: float) -> float:
    """Epsilon-safe signed division (matching MINLPLIB version)"""
    if den == 0.0:
        den_adj = eps
    else:
        den_adj = den + np.sign(den) * eps
    return float(num / den_adj)


def _bounds_exist_flag(inst: Any) -> int:
    """
    Check if bounds exist (natural bounds [0,1])
    Returns 1 if l=0 and u=1 (natural bounds exist)
    """
    be = _get(inst, "bounds_exist", None)
    if be is not None:
        return int(bool(be))

    l = np.asarray(_get(inst, "l", []), dtype=float).reshape(-1)
    u = np.asarray(_get(inst, "u", []), dtype=float).reshape(-1)

    if l.size == 0 or u.size == 0 or l.size != u.size:
        return 0

    return int(np.allclose(l, 0.0) and np.allclose(u, 1.0))


# ============================================================
# SPARSITY PATTERN DETECTION
# ============================================================

def detect_sparsity_patterns(A, tol=1e-8):
    """
    Detect sparsity patterns:
    Diagonal, Hollow, Bipartite, Tree, Chordal, Planar
    """
    patterns = {}
    A = _sym(A)
    n = A.shape[0]

    off_diag = A - np.diag(np.diag(A))
    patterns['Diagonal'] = bool(np.allclose(off_diag, 0, atol=tol))
    patterns['Hollow'] = bool(np.allclose(np.diag(A), 0, atol=tol))

    patterns['Bipartite'] = False
    patterns['Tree'] = False
    patterns['Chordal'] = False
    patterns['Planar'] = False

    mask = (np.abs(np.triu(off_diag, 1)) > tol)

    if np.any(mask):
        rows, cols = np.where(mask)
        G = nx.Graph()
        G.add_nodes_from(range(n))
        G.add_edges_from(zip(rows.tolist(), cols.tolist()))

        patterns['Bipartite'] = bool(bipartite.is_bipartite(G))
        patterns['Tree'] = bool(nx.is_tree(G))

        planar, _ = nx.check_planarity(G)
        patterns['Planar'] = bool(planar)

        patterns['Chordal'] = bool(nx.is_chordal(G))

    return patterns


# ============================================================
# STATISTICAL FEATURES
# ============================================================

def compute_statistics(vector):
    """
    Compute statistics for a vector.
    ✅ mu3 is RAW 3rd central moment (NOT standardized) - matching MINLPLIB version
    ✅ CV uses epsilon-safe signed division
    """
    if vector is None or len(vector) == 0:
        return {
            "min": 0.0, "max": 0.0, "avg": 0.0, "mu3": 0.0,
            "IQR": 0.0, "Pout": 0.0, "CV": 0.0
        }

    v = np.asarray(vector, dtype=float).reshape(-1)
    mean_val = float(np.mean(v))
    std_val  = float(np.std(v))

    # ✅ RAW 3rd central moment (matching MINLPLIB version)
    mu3 = float(np.mean((v - mean_val) ** 3)) if v.size > 1 else 0.0

    q1, q3 = np.percentile(v, [25, 75])
    iqr = float(q3 - q1)

    if iqr > 0:
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        pout = float(np.mean((v < lower) | (v > upper)))
    else:
        pout = 0.0

    # ✅ Epsilon-safe signed CV (matching MINLPLIB version)
    cv = _safe_signed_div(std_val, mean_val, EPS_CV)

    return {
        "min": float(np.min(v)),
        "max": float(np.max(v)),
        "avg": float(mean_val),
        "mu3": float(mu3),
        "IQR": float(iqr),
        "Pout": float(pout),
        "CV": float(cv),
    }


# ============================================================
# EXTRA FEATURES
# ============================================================

def _log10_cond_from_eigs(eigs, eps=EPS_EIG):
    """Returns log10(condition number) clamped to [0, 16]"""
    a = np.abs(np.asarray(eigs, dtype=float))
    amax = float(np.max(a)) if a.size else 1.0
    nz = a[a > eps]

    if nz.size == 0:
        cond = 1e16
    else:
        amin = float(np.min(nz))
        cond = amax / (amin + EPS_NUM)

    cond = max(1.0, min(cond, 1e16))
    return float(np.log10(cond))


def _log10_diag_off_energy_ratio(A, tiny=EPS_NUM):
    """Returns log10(diag/off ratio) clamped to [-6, 16]"""
    A = _sym(A)
    d = np.diag(A)
    diag_e = float(np.sum(d * d))
    tot_e = float(np.sum(A * A))
    off_e = max(tot_e - diag_e, 0.0)

    if off_e < tiny:
        ratio = 1e16
    elif diag_e < tiny:
        ratio = 1e-6
    else:
        ratio = diag_e / (off_e + tiny)

    ratio = max(1e-6, min(ratio, 1e16))
    return float(np.log10(ratio))


def _commutator_norm(A, B, tiny=EPS_NUM):
    """Normalized Frobenius norm of [A,B]"""
    A = _sym(A)
    B = _sym(B)
    AB = A @ B
    BA = B @ A
    C = AB - BA
    num = float(np.linalg.norm(C, ord='fro'))
    den = float(np.linalg.norm(A, ord='fro') * np.linalg.norm(B, ord='fro'))
    return num / (den + tiny)


def _compute_commutator_features(A_list, max_pairs=40):
    """
    Compute commutator features.
    ✅ comm_min DROPPED - only comm_mean / comm_max kept (matching MINLPLIB pruned version)
    """
    if len(A_list) < 2:
        return {"comm_mean": 0.0, "comm_max": 0.0}

    mats = [_sym(A) for A in A_list]
    norms = []

    for k in range(1, len(mats)):
        norms.append(_commutator_norm(mats[0], mats[k]))

    if len(mats) > 2:
        total_pairs = (len(mats) - 1) * (len(mats) - 2) // 2
        if total_pairs <= max_pairs:
            for i in range(1, len(mats)):
                for j in range(i + 1, len(mats)):
                    norms.append(_commutator_norm(mats[i], mats[j]))
        else:
            all_pairs = [(i, j) for i in range(1, len(mats)) for j in range(i + 1, len(mats))]
            sampled = random.sample(all_pairs, min(max_pairs, len(all_pairs)))
            for i, j in sampled:
                norms.append(_commutator_norm(mats[i], mats[j]))

    if len(norms) == 0:
        return {"comm_mean": 0.0, "comm_max": 0.0}

    norms = np.asarray(norms, dtype=float)
    # ✅ comm_min removed (matching MINLPLIB pruned version)
    return {
        "comm_mean": float(np.mean(norms)),
        "comm_max": float(np.max(norms)),
    }


def _sample_abs_entries(arr, cap=200000, tiny=EPS_NUM):
    """Sample absolute values from array"""
    v = np.abs(np.asarray(arr, dtype=float).ravel())
    v = v[np.isfinite(v)]
    v = v[v > tiny]

    if v.size == 0:
        return v
    if v.size <= cap:
        return v

    step = int(np.ceil(v.size / cap))
    return v[::step]


def _scale_features_from_values(v, tiny=EPS_NUM):
    """Compute scale features from values"""
    if v.size == 0:
        return {
            "scale_max_abs": 0.0,
            "scale_min_pos_abs": 0.0,
            "scale_median_abs": 0.0,
            "scale_log10_range": 0.0,
            "scale_max_over_median": 0.0,
            "scale_p95_over_p05": 0.0,
        }

    vmax = float(np.max(v))
    vmin = float(np.min(v))
    vmed = float(np.median(v))
    p05 = float(np.percentile(v, 5)) if v.size >= 20 else vmin
    p95 = float(np.percentile(v, 95)) if v.size >= 20 else vmax

    return {
        "scale_max_abs": vmax,
        "scale_min_pos_abs": vmin,
        "scale_median_abs": vmed,
        "scale_log10_range": float(np.log10((vmax + tiny) / (vmin + tiny))),
        "scale_max_over_median": float((vmax + tiny) / (vmed + tiny)),
        "scale_p95_over_p05": float((p95 + tiny) / (p05 + tiny)),
    }


# ============================================================
# fDD (Fully Dimension-Dependent)
# ============================================================

def extract_features_fDD(instances, tol=1e-8, extra_feature_sets=()):
    """Fully Dimension-Dependent setup"""
    extra_feature_sets = set(extra_feature_sets or [])

    # Apply pruning policy globals
    if not INCLUDE_DIAG_OFF_RATIO:
        extra_feature_sets.discard("diag_off_ratio")
    if not INCLUDE_SCALE_FEATURES:
        extra_feature_sets.discard("scale")

    rows = []
    sparsity_patterns = ['Diagonal', 'Hollow', 'Bipartite', 'Tree', 'Chordal', 'Planar']

    for inst0 in instances:
        inst = _to_dict(inst0)
        A_list = _get(inst, 'A_list', [])

        if not A_list:
            continue

        row = {
            'id': int(_get(inst, 'instance_id', _get(inst, 'id', -1))),
            'BoundsExist': _bounds_exist_flag(inst)
        }

        for k, A in enumerate(A_list):
            A = _sym(A)
            eigs = np.linalg.eigvalsh(A)

            for j, eig in enumerate(eigs):
                row[f'lambda_{j+1}_A{k}'] = float(eig)

            row[f'eta_A{k}'] = int(np.sum(eigs < -tol))
            row[f'rho_A{k}'] = int(np.linalg.matrix_rank(A))

            patterns = detect_sparsity_patterns(A, tol=tol)
            for pattern in sparsity_patterns:
                row[f'P{pattern[0]}_A{k}'] = int(patterns[pattern])

            # ✅ cond: controlled by both caller flag AND INCLUDE_COND_A0 global
            if "cond" in extra_feature_sets and INCLUDE_COND_A0:
                row[f'log10_cond_A{k}'] = _log10_cond_from_eigs(eigs)

            if "diag_off_ratio" in extra_feature_sets:
                row[f'log10_diagoff_A{k}'] = _log10_diag_off_energy_ratio(A)

        if "commutator" in extra_feature_sets:
            comm = _compute_commutator_features(A_list, max_pairs=40)
            row.update(comm)

        if "scale" in extra_feature_sets:
            b_list = _get(inst, 'b_list', [])
            c_list = _get(inst, 'c_list', [])

            vals = []
            for A in A_list:
                vals.append(_sample_abs_entries(A, cap=200000))
            if b_list:
                for b in b_list:
                    vals.append(_sample_abs_entries(b, cap=200000))
            if c_list:
                vals.append(_sample_abs_entries(np.asarray(c_list, dtype=float), cap=200000))

            v = np.concatenate([x for x in vals if x.size > 0], axis=0) if any(x.size > 0 for x in vals) else np.array([])
            sfeat = _scale_features_from_values(v)
            row.update(sfeat)

        rows.append(row)

    return pd.DataFrame(rows)


# ============================================================
# sDD (Semi Dimension-Dependent)
# ============================================================

def extract_features_sDD(instances, tol=1e-8, extra_feature_sets=()):
    """Semi Dimension-Dependent setup"""
    extra_feature_sets = set(extra_feature_sets or [])

    # Apply pruning policy globals
    if not INCLUDE_DIAG_OFF_RATIO:
        extra_feature_sets.discard("diag_off_ratio")
    if not INCLUDE_SCALE_FEATURES:
        extra_feature_sets.discard("scale")

    rows = []
    sparsity_patterns = ['Diagonal', 'Hollow', 'Bipartite', 'Tree', 'Chordal', 'Planar']

    for inst0 in instances:
        inst = _to_dict(inst0)
        A_list = _get(inst, 'A_list', [])

        if not A_list:
            continue

        n = int(np.asarray(A_list[0]).shape[0])

        row = {
            'id': int(_get(inst, 'instance_id', _get(inst, 'id', -1))),
            'n': n,
            'BoundsExist': _bounds_exist_flag(inst)
        }

        for k, A in enumerate(A_list):
            A = _sym(A)
            eigs = np.linalg.eigvalsh(A)

            neg_count = np.sum(eigs < -tol)
            row[f'eta_A{k}_over_n'] = float(neg_count / n)

            neg_eigs = eigs[eigs < -tol]
            total_abs_sum = float(np.sum(np.abs(eigs)))
            if total_abs_sum > 0:
                row[f'theta_A{k}'] = float(np.sum(np.abs(neg_eigs)) / total_abs_sum)
            else:
                row[f'theta_A{k}'] = 0.0

            row[f'lambda_min_A{k}'] = float(np.min(eigs))
            row[f'lambda_max_A{k}'] = float(np.max(eigs))
            row[f'rho_A{k}_over_n'] = float(np.linalg.matrix_rank(A) / n)

            patterns = detect_sparsity_patterns(A, tol=tol)
            for pattern in sparsity_patterns:
                row[f'P{pattern[0]}_A{k}'] = int(patterns[pattern])

            # ✅ cond: controlled by both caller flag AND INCLUDE_COND_A0 global
            if "cond" in extra_feature_sets and INCLUDE_COND_A0:
                row[f'log10_cond_A{k}'] = _log10_cond_from_eigs(eigs)

            if "diag_off_ratio" in extra_feature_sets:
                row[f'log10_diagoff_A{k}'] = _log10_diag_off_energy_ratio(A)

        if "commutator" in extra_feature_sets:
            comm = _compute_commutator_features(A_list, max_pairs=40)
            row.update(comm)

        if "scale" in extra_feature_sets:
            b_list = _get(inst, 'b_list', [])
            c_list = _get(inst, 'c_list', [])

            vals = []
            for A in A_list:
                vals.append(_sample_abs_entries(A, cap=200000))
            if b_list:
                for b in b_list:
                    vals.append(_sample_abs_entries(b, cap=200000))
            if c_list:
                vals.append(_sample_abs_entries(np.asarray(c_list, dtype=float), cap=200000))

            v = np.concatenate([x for x in vals if x.size > 0], axis=0) if any(x.size > 0 for x in vals) else np.array([])
            sfeat = _scale_features_from_values(v)
            row.update(sfeat)

        rows.append(row)

    return pd.DataFrame(rows)


# ============================================================
# DI (Dimension-Independent)
# ============================================================

def extract_features_DI(instances, tol=1e-8, extra_feature_sets=(), comm_max_pairs=40, scale_cap=200000):
    """Dimension-Independent setup"""
    extra_feature_sets = set(extra_feature_sets or [])

    # Apply pruning policy globals
    if not INCLUDE_DIAG_OFF_RATIO:
        extra_feature_sets.discard("diag_off_ratio")
    if not INCLUDE_SCALE_FEATURES:
        extra_feature_sets.discard("scale")

    rows = []
    sparsity_patterns = ["Diagonal", "Hollow", "Bipartite", "Tree", "Chordal", "Planar"]
    stat_names = ["min", "max", "avg", "mu3", "IQR", "Pout", "CV"]

    for inst0 in instances:
        inst = _to_dict(inst0)

        A_list = _get(inst, "A_list", [])
        b_list = _get(inst, "b_list", [])
        c_list = _get(inst, "c_list", [])

        if A_list is None or len(A_list) == 0:
            continue

        n = int(np.asarray(A_list[0]).shape[0])
        m = int(len(A_list) - 1)

        row = {
            "id": int(_get(inst, "instance_id", _get(inst, "id", -1))),
            "n": n,
            "m": m,
            "BoundsExist": _bounds_exist_flag(inst),
        }

        A0 = _sym(A_list[0])
        eigs0 = np.linalg.eigvalsh(A0)

        neg_count0 = np.sum(eigs0 < -tol)
        row["eta_A0_over_n"] = float(neg_count0 / n)

        neg_eigs0 = eigs0[eigs0 < -tol]
        total_abs_sum0 = float(np.sum(np.abs(eigs0)))
        row["theta_A0"] = float(np.sum(np.abs(neg_eigs0)) / total_abs_sum0) if total_abs_sum0 > 0 else 0.0

        row["lambda_min_A0"] = float(np.min(eigs0))
        row["lambda_max_A0"] = float(np.max(eigs0))
        row["rho_A0_over_n"] = float(np.linalg.matrix_rank(A0) / n)

        patterns0 = detect_sparsity_patterns(A0, tol=tol)
        for pattern in sparsity_patterns:
            row[f"P{pattern[0]}_A0"] = int(patterns0[pattern])

        # ✅ A0 cond: gated by both caller flag AND INCLUDE_COND_A0 global
        if "cond" in extra_feature_sets and INCLUDE_COND_A0:
            row["log10_cond_A0"] = _log10_cond_from_eigs(eigs0)

        if "diag_off_ratio" in extra_feature_sets:
            row["log10_diagoff_A0"] = _log10_diag_off_energy_ratio(A0)

        if m > 0:
            Xi = []
            Theta = []
            Lambda_min = []
            Lambda_max = []
            Rho = []

            LogConds = []
            LogDiagOffs = []

            sparsity_vectors = {pattern: [] for pattern in sparsity_patterns}
            scale_vals = []

            if "scale" in extra_feature_sets:
                scale_vals.append(_sample_abs_entries(A0, cap=scale_cap))

            for k in range(1, len(A_list)):
                Ak = _sym(A_list[k])
                eigs = np.linalg.eigvalsh(Ak)

                neg_count = np.sum(eigs < -tol)
                Xi.append(float(neg_count / n))

                neg_eigs = eigs[eigs < -tol]
                total_abs_sum = float(np.sum(np.abs(eigs)))
                theta_val = float(np.sum(np.abs(neg_eigs)) / total_abs_sum) if total_abs_sum > 0 else 0.0
                Theta.append(theta_val)

                Lambda_min.append(float(np.min(eigs)))
                Lambda_max.append(float(np.max(eigs)))
                Rho.append(float(np.linalg.matrix_rank(Ak) / n))

                if "cond" in extra_feature_sets:
                    LogConds.append(_log10_cond_from_eigs(eigs))

                if "diag_off_ratio" in extra_feature_sets:
                    LogDiagOffs.append(_log10_diag_off_energy_ratio(Ak))

                if "scale" in extra_feature_sets:
                    scale_vals.append(_sample_abs_entries(Ak, cap=scale_cap))

                patterns = detect_sparsity_patterns(Ak, tol=tol)
                for pattern in sparsity_patterns:
                    sparsity_vectors[pattern].append(int(patterns[pattern]))

            convex_count = sum(1 for t in Theta if t == 0.0)
            concave_count = sum(1 for t in Theta if t == 1.0)
            row["g0_Theta"] = float(convex_count / m)
            row["g1_Theta"] = float(concave_count / m)

            properties = {
                "Xi": Xi,
                "Theta": Theta,
                "Lambda_min": Lambda_min,
                "Lambda_max": Lambda_max,
                "Rho": Rho,
            }

            for prop_name, vec in properties.items():
                stats = compute_statistics(vec)
                for stat_name in stat_names:
                    row[f"h_{stat_name}_{prop_name}"] = stats[stat_name]

            # ✅ LogCond stats: only keep whitelisted stats (KEEP_LOGCOND_STATS)
            if "cond" in extra_feature_sets:
                cstats = compute_statistics(LogConds)
                for stat_name in KEEP_LOGCOND_STATS:
                    row[f"h_{stat_name}_LogCond"] = cstats[stat_name]

            # ✅ LogDiagOff stats: only keep whitelisted stats (KEEP_LOGDIAGOFF_STATS)
            if "diag_off_ratio" in extra_feature_sets:
                rstats = compute_statistics(LogDiagOffs)
                for stat_name in KEEP_LOGDIAGOFF_STATS:
                    row[f"h_{stat_name}_LogDiagOff"] = rstats[stat_name]

            for pattern in sparsity_patterns:
                row[f"g{pattern[0]}_Phi{pattern[0]}"] = float(sum(sparsity_vectors[pattern]) / m)

            if "scale" in extra_feature_sets:
                if b_list is not None:
                    for bk in b_list:
                        scale_vals.append(_sample_abs_entries(bk, cap=scale_cap))

                if c_list is not None:
                    scale_vals.append(_sample_abs_entries(np.asarray(c_list, dtype=float), cap=scale_cap))

                if len(scale_vals):
                    v = np.concatenate([x for x in scale_vals if x.size > 0], axis=0) if any(x.size > 0 for x in scale_vals) else np.array([])
                else:
                    v = np.array([])

                sfeat = _scale_features_from_values(v)
                row.update(sfeat)

        else:
            row["g0_Theta"] = 0.0
            row["g1_Theta"] = 0.0

            for prop_name in ["Xi", "Theta", "Lambda_min", "Lambda_max", "Rho"]:
                for stat_name in stat_names:
                    row[f"h_{stat_name}_{prop_name}"] = 0.0

            if "cond" in extra_feature_sets:
                for stat_name in KEEP_LOGCOND_STATS:
                    row[f"h_{stat_name}_LogCond"] = 0.0

            if "diag_off_ratio" in extra_feature_sets:
                for stat_name in KEEP_LOGDIAGOFF_STATS:
                    row[f"h_{stat_name}_LogDiagOff"] = 0.0

            for pattern in sparsity_patterns:
                row[f"g{pattern[0]}_Phi{pattern[0]}"] = 0.0

            if "scale" in extra_feature_sets:
                sfeat = _scale_features_from_values(np.array([]))
                row.update(sfeat)

        if "commutator" in extra_feature_sets:
            comm = _compute_commutator_features(A_list, max_pairs=comm_max_pairs)
            row.update(comm)

        rows.append(row)

    df = pd.DataFrame(rows)

    return df


# ============================================================
# MAIN WRAPPER
# ============================================================

def extract_features_all_setups(
    instances,
    setups=['fDD', 'sDD', 'DI'],
    tol=1e-8,
    extra_feature_sets=(),
    comm_max_pairs=40,
    scale_cap=200000,
):
    """
    Extract features for all requested setups

    Args:
        instances: List of instances
        setups: List of setups ('fDD', 'sDD', 'DI')
        tol: Tolerance
        extra_feature_sets: ("cond", "diag_off_ratio", "commutator", "scale")
                            Note: diag_off_ratio and scale are gated by globals
                            INCLUDE_DIAG_OFF_RATIO / INCLUDE_SCALE_FEATURES
        comm_max_pairs: Max constraint pairs for commutator
        scale_cap: Max entries for scale features

    Returns:
        Dictionary of DataFrames
    """
    results = {}

    if 'fDD' in setups:
        print("Extracting fDD features...")
        results['fDD'] = extract_features_fDD(instances, tol=tol, extra_feature_sets=extra_feature_sets)
        print(f"  ✅ fDD: {results['fDD'].shape}")

    if 'sDD' in setups:
        print("Extracting sDD features...")
        results['sDD'] = extract_features_sDD(instances, tol=tol, extra_feature_sets=extra_feature_sets)
        print(f"  ✅ sDD: {results['sDD'].shape}")

    if 'DI' in setups:
        print("Extracting DI features...")
        results['DI'] = extract_features_DI(
            instances,
            tol=tol,
            extra_feature_sets=extra_feature_sets,
            comm_max_pairs=comm_max_pairs,
            scale_cap=scale_cap,
        )
        print(f"  ✅ DI: {results['DI'].shape}")

    return results


# ============================================================
# MAIN RUNNER
# ============================================================

if __name__ == '__main__':

    PKL_PATH = 'instances_5_1_50.pkl'
    OUTPUT_PREFIX = 'features'
    OUTPUT_SUFFIX = '5_1_50'

    SETUPS = ['fDD', 'sDD', 'DI']

    # ✅ RECOMMENDATION: Only keep condition number
    # comm_min is dropped automatically now (only comm_mean/comm_max emitted)
    # log10_cond_A0 is dropped by default (INCLUDE_COND_A0 = False)
    # diag_off_ratio is dropped by default (INCLUDE_DIAG_OFF_RATIO = False)
    # scale is dropped by default (INCLUDE_SCALE_FEATURES = False)

    EXTRA_FEATURES = ("cond", "commutator")

    # To include all (still subject to global switches above):
    # EXTRA_FEATURES = ("cond", "diag_off_ratio", "commutator", "scale")

    # To exclude all:
    # EXTRA_FEATURES = ()

    print("=" * 60)
    print("FEATURE EXTRACTION - ALL SETUPS (UPDATED)")
    print("=" * 60)
    print(f"Input:  {PKL_PATH}")
    print(f"Setups: {SETUPS}")
    print(f"Extra:  {EXTRA_FEATURES if EXTRA_FEATURES else 'None'}")
    print(f"✅ mu3 = raw 3rd central moment (NOT standardized)")
    print(f"✅ CV  = epsilon-safe signed division")
    print(f"✅ comm_min dropped (comm_mean/comm_max only)")
    print(f"✅ INCLUDE_COND_A0        = {INCLUDE_COND_A0}")
    print(f"✅ INCLUDE_DIAG_OFF_RATIO = {INCLUDE_DIAG_OFF_RATIO}")
    print(f"✅ INCLUDE_SCALE_FEATURES = {INCLUDE_SCALE_FEATURES}")
    print(f"✅ KEEP_LOGCOND_STATS     = {KEEP_LOGCOND_STATS}")
    print("=" * 60)

    with open(PKL_PATH, 'rb') as f:
        instances = pickle.load(f)

    if instances and hasattr(instances[0], '__dict__'):
        instances = [inst if isinstance(inst, dict) else inst.__dict__ for inst in instances]

    print(f"📦 Loaded {len(instances)} instances")

    feature_dfs = extract_features_all_setups(
        instances,
        setups=SETUPS,
        tol=1e-8,
        extra_feature_sets=EXTRA_FEATURES,
        comm_max_pairs=40,
        scale_cap=200000,
    )

    print("\n" + "=" * 60)
    print("SAVING RESULTS")
    print("=" * 60)

    for setup_name, df in feature_dfs.items():
        filename = f'{OUTPUT_PREFIX}_{setup_name}_{OUTPUT_SUFFIX}.xlsx'
        df.to_excel(filename, index=False)
        print(f"✅ {filename}: {df.shape} ({df.shape[1]-1} features)")

    print("\n" + "=" * 60)
    print("✅ COMPLETE")
    print("=" * 60)

FEATURE EXTRACTION - ALL SETUPS (UPDATED)
Input:  instances_5_1_50.pkl
Setups: ['fDD', 'sDD', 'DI']
Extra:  ('cond', 'commutator')
✅ mu3 = raw 3rd central moment (NOT standardized)
✅ CV  = epsilon-safe signed division
✅ comm_min dropped (comm_mean/comm_max only)
✅ INCLUDE_COND_A0        = False
✅ INCLUDE_DIAG_OFF_RATIO = False
✅ INCLUDE_SCALE_FEATURES = False
✅ KEEP_LOGCOND_STATS     = ('avg', 'IQR', 'mu3', 'CV')
📦 Loaded 50000 instances
Extracting fDD features...
  ✅ fDD: (50000, 30)
Extracting sDD features...
  ✅ sDD: (50000, 27)
Extracting DI features...
  ✅ DI: (50000, 64)

SAVING RESULTS
✅ features_fDD_5_1_50.xlsx: (50000, 30) (29 features)
✅ features_sDD_5_1_50.xlsx: (50000, 27) (26 features)
✅ features_DI_5_1_50.xlsx: (50000, 64) (63 features)

✅ COMPLETE
